In [6]:

"""
Roof Texture Model Inference and Visualization

This script consolidates the functionality of a Jupyter notebook into a
single Python file. It loads a trained roof texture classification model,
performs inference on image tiles, and visualizes predictions.
"""

import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import matplotlib.pyplot as plt
from PIL import Image  # noqa: F401  (kept for parity with notebook)
import warnings

warnings.filterwarnings('ignore')
os.environ['OPENCV_LOG_LEVEL'] = 'SILENT'
print("✅ Libraries imported successfully!")

# ------------------------------
# Model Architecture
# ------------------------------

class RoofTextureClassifier(nn.Module):
    """Roof texture classification model - Original version"""
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)


class ConservativeRoofClassifier(nn.Module):
    """Conservative model with batch normalization"""
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)


class ImprovedRoofTextureClassifier(nn.Module):
    """Improved CNN model with better regularization"""
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

print("✅ Model classes defined successfully!")

# ------------------------------
# Configuration and Setup
# ------------------------------

class InferenceConfig:
    """Configuration for inference"""
    MODEL_PATHS = {
        'original': "best_roof_classifier.pth",
    }
    IMAGE_SIZE = 224
    TILE_SIZE = 256
    OVERLAP = 64
    BATCH_SIZE = 16
    CLASS_LABELS = ['smooth', 'average', 'rough']
    CLASS_COLORS = {
        'smooth': (83, 217, 56),    # Green
        'average': (252, 240, 3),   # Yellow
        'rough': (255, 0, 0)        # Red
    }
    SAMPLE_PATHS = [
        "/home/student/sky-scan/data/patch",
        "/Users/shahaziz/Learning/sky-scan/data/patch",
        "../data/patch",
        "./data/patch"
    ]

config = InferenceConfig()

def get_device():
    """Get available device"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name()}")
    return device

device = get_device()
print("✅ Configuration loaded successfully!")

# ------------------------------
# Model Loading
# ------------------------------

def auto_detect_model_type(checkpoint):
    """Automatically detect model type from checkpoint"""
    model_class = checkpoint.get('model_class', '')
    if 'Conservative' in model_class:
        return 'conservative'
    elif 'Improved' in model_class:
        return 'improved'
    else:
        return 'original'

def load_model(model_path, num_classes=3, model_type='auto'):
    """Load trained model from checkpoint with auto-detection"""
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")
    checkpoint = torch.load(model_path, map_location=device)
    if model_type == 'auto':
        model_type = auto_detect_model_type(checkpoint)
    if model_type == 'conservative':
        model = ConservativeRoofClassifier(num_classes)
    elif model_type == 'improved':
        model = ImprovedRoofTextureClassifier(num_classes)
    else:
        model = RoofTextureClassifier(num_classes)
    try:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Model loaded from {model_path}")
        print(f"📝 Model type: {model_type} ({checkpoint.get('model_class', 'Unknown')})")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise
    model.to(device)
    model.eval()
    return model, model_type

def find_and_load_model():
    """Find and load the first available model"""
    for model_name, model_path in config.MODEL_PATHS.items():
        if os.path.exists(model_path):
            try:
                model, model_type = load_model(model_path)
                return model, model_type, model_path
            except Exception as e:
                print(f"⚠️ Failed to load {model_name} model: {e}")
                continue
    return None, None, None

print("✅ Model loading functions defined!")

# ------------------------------
# Image Processing
# ------------------------------

def get_inference_transform(image_size=224):
    """Get transform for inference"""
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

def extract_tiles(image, tile_size=256, overlap=64):
    """Extract overlapping tiles from a large image"""
    h, w = image.shape[:2]
    tiles = []
    positions = []
    stride = tile_size - overlap
    y_positions = list(range(0, h - tile_size + 1, stride))
    x_positions = list(range(0, w - tile_size + 1, stride))
    if y_positions and y_positions[-1] + tile_size < h:
        y_positions.append(h - tile_size)
    if x_positions and x_positions[-1] + tile_size < w:
        x_positions.append(w - tile_size)
    for y in y_positions:
        for x in x_positions:
            tile = image[y:y+tile_size, x:x+tile_size]
            if tile.shape[:2] == (tile_size, tile_size):
                tiles.append(tile)
                positions.append((x, y))
    print(f"📐 Extracted {len(tiles)} tiles of size {tile_size}x{tile_size}")
    return tiles, positions

def preprocess_tiles(tiles, transform):
    """Preprocess tiles for model inference"""
    processed_tiles = []
    for tile in tiles:
        if len(tile.shape) == 3:
            if tile.shape[2] == 3:
                tile_rgb = tile
            else:
                tile_rgb = tile
        else:
            tile_rgb = cv2.cvtColor(tile, cv2.COLOR_GRAY2RGB)
        try:
            tile_tensor = transform(tile_rgb)
            processed_tiles.append(tile_tensor)
        except Exception as e:
            print(f"⚠️ Error processing tile: {e}")
            continue
    return torch.stack(processed_tiles) if processed_tiles else torch.empty(0)

transform = get_inference_transform(config.IMAGE_SIZE)
print("✅ Image processing functions defined!")

# ------------------------------
# Inference
# ------------------------------

def predict_tiles(model, tiles_tensor, batch_size=16):
    """Predict labels for tiles using the trained model"""
    if len(tiles_tensor) == 0:
        return [], []
    model.eval()
    predictions = []
    confidences = []
    with torch.no_grad():
        for i in range(0, len(tiles_tensor), batch_size):
            batch = tiles_tensor[i:i+batch_size].to(device)
            try:
                outputs = model(batch)
                probabilities = F.softmax(outputs, dim=1)
                confidence_scores, predicted_classes = torch.max(probabilities, 1)
                predictions.extend(predicted_classes.cpu().numpy())
                confidences.extend(confidence_scores.cpu().numpy())
            except Exception as e:
                print(f"⚠️ Error in batch prediction: {e}")
                continue
    return predictions, confidences

def get_class_name(class_idx):
    """Convert class index to class name"""
    if class_idx < len(config.CLASS_LABELS):
        return config.CLASS_LABELS[class_idx]
    return 'unknown'

print("✅ Inference functions defined!")

# ------------------------------
# Visualization
# ------------------------------

def create_colored_tile(tile, class_name, confidence, alpha=0.4):
    """Create a colored tile based on prediction"""
    color = config.CLASS_COLORS.get(class_name, (128, 128, 128))
    overlay = np.full_like(tile, color, dtype=np.uint8)
    alpha_conf = alpha * float(confidence)
    colored_tile = cv2.addWeighted(tile, 1-alpha_conf, overlay, alpha_conf, 0)
    return colored_tile

def create_prediction_map(image_shape, positions, predictions, confidences, tile_size=256):
    """Create a clean prediction map without the original image"""
    h, w = image_shape[:2]
    prediction_map = np.zeros((h, w, 3), dtype=np.uint8)
    confidence_map = np.zeros((h, w), dtype=np.float32)
    for (x, y), pred_idx, confidence in zip(positions, predictions, confidences):
        class_name = get_class_name(int(pred_idx))
        color = config.CLASS_COLORS.get(class_name, (128, 128, 128))
        end_x = min(x + tile_size, w)
        end_y = min(y + tile_size, h)
        current_confidence = confidence_map[y:end_y, x:end_x]
        mask = float(confidence) < current_confidence
        prediction_map[y:end_y, x:end_x][~mask] = color
        confidence_map[y:end_y, x:end_x] = np.maximum(confidence_map[y:end_y, x:end_x], float(confidence))
    return prediction_map

def merge_tiles_with_predictions(original_image, tiles, positions, predictions, confidences,
                                tile_size=256, alpha=0.4):
    """Merge colored tiles back into a complete image"""
    result_image = original_image.copy()
    h, w = original_image.shape[:2]
    for tile, (x, y), pred_idx, confidence in zip(tiles, positions, predictions, confidences):
        class_name = get_class_name(int(pred_idx))
        colored_tile = create_colored_tile(tile, class_name, float(confidence), alpha)
        end_x = min(x + tile_size, w)
        end_y = min(y + tile_size, h)
        tile_h = end_y - y
        tile_w = end_x - x
        result_image[y:end_y, x:end_x] = colored_tile[:tile_h, :tile_w]
    return result_image

print("✅ Visualization functions defined!")

# ------------------------------
# Statistics and Analysis
# ------------------------------

def analyze_predictions(predictions, confidences):
    """Analyze prediction statistics"""
    if not predictions:
        return {}
    stats = {}
    total_tiles = len(predictions)
    for i, class_name in enumerate(config.CLASS_LABELS):
        class_mask = np.array(predictions) == i
        count = int(np.sum(class_mask))
        avg_confidence = float(np.mean(np.array(confidences)[class_mask])) if count > 0 else 0.0
        stats[class_name] = {
            'count': count,
            'percentage': (count / total_tiles * 100.0) if total_tiles > 0 else 0.0,
            'avg_confidence': avg_confidence
        }
    return stats

def plot_statistics(stats):
    """Plot prediction statistics"""
    if not stats:
        print("⚠️ No statistics to plot")
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    classes = list(stats.keys())
    counts = [stats[cls]['count'] for cls in classes]
    colors = [tuple(c/255.0 for c in config.CLASS_COLORS[cls]) for cls in classes]
    ax1.pie(counts, labels=classes, colors=colors, autopct='%1.1f%%', startangle=90)
    ax1.set_title('Roof Texture Distribution', fontsize=14, fontweight='bold')
    confidences = [stats[cls]['avg_confidence'] for cls in classes]
    bars = ax2.bar(classes, confidences, color=colors, alpha=0.8, edgecolor='black')
    ax2.set_title('Average Confidence by Class', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Average Confidence', fontsize=12)
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    for bar, conf in zip(bars, confidences):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{conf:.3f}', ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✅ Analysis functions defined!")

# ------------------------------
# Main Processing
# ------------------------------

def process_image(image_path, model, save_results=True, output_dir="inference_results"):
    """Process a single image: extract tiles, predict, and visualize"""
    if model is None:
        print("❌ No model loaded. Please train a model first.")
        return None
    print(f"\n🖼️ Processing image: {os.path.basename(image_path)}")
    if not os.path.exists(image_path):
        print(f"❌ Image file not found: {image_path}")
        return None
    image = cv2.imread(image_path)
    if image is None:
        print(f"❌ Failed to load image: {image_path}")
        return None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"📏 Image shape: {image_rgb.shape}")
    print("🔪 Extracting tiles...")
    tiles, positions = extract_tiles(image_rgb, config.TILE_SIZE, config.OVERLAP)
    if not tiles:
        print("❌ No tiles extracted from image")
        return None
    print("🔄 Preprocessing tiles...")
    tiles_tensor = preprocess_tiles(tiles, transform)
    if len(tiles_tensor) == 0:
        print("❌ No valid tiles after preprocessing")
        return None
    print("🧠 Running inference...")
    predictions, confidences = predict_tiles(model, tiles_tensor, config.BATCH_SIZE)
    if not predictions:
        print("❌ No predictions generated")
        return None
    stats = analyze_predictions(predictions, confidences)
    print("\n📊 Prediction Statistics:")
    for class_name, stat in stats.items():
        print(f"  🔸 {class_name.capitalize()}: {stat['count']} tiles ({stat['percentage']:.1f}%), "
              f"confidence: {stat['avg_confidence']:.3f}")
    print("\n🎨 Creating visualizations...")
    colored_result = merge_tiles_with_predictions(
        image_rgb, tiles, positions, predictions, confidences,
        config.TILE_SIZE, alpha=0.5
    )
    prediction_map = create_prediction_map(
        image_rgb.shape, positions, predictions, confidences, config.TILE_SIZE
    )
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].imshow(image_rgb); axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold'); axes[0, 0].axis('off')
    axes[0, 1].imshow(colored_result); axes[0, 1].set_title('Predictions Overlay', fontsize=14, fontweight='bold'); axes[0, 1].axis('off')
    axes[1, 0].imshow(prediction_map); axes[1, 0].set_title('Prediction Map', fontsize=14, fontweight='bold'); axes[1, 0].axis('off')
    axes[1, 1].axis('off')
    legend_y = 0.9
    axes[1, 1].text(0.1, legend_y, 'Legend & Statistics', fontsize=16, fontweight='bold')
    legend_y -= 0.1
    for class_name, color in config.CLASS_COLORS.items():
        color_normalized = tuple(c/255.0 for c in color)
        axes[1, 1].add_patch(plt.Rectangle((0.1, legend_y), 0.08, 0.06,
                                           facecolor=color_normalized, edgecolor='black'))
        stat = stats[class_name]
        axes[1, 1].text(0.22, legend_y + 0.03,
                        f"{class_name.capitalize()}: {stat['count']} tiles ({stat['percentage']:.1f}%)",
                        fontsize=12, va='center')
        legend_y -= 0.08
    legend_y -= 0.05
    avg_conf = float(np.mean(confidences))
    axes[1, 1].text(0.1, legend_y, f'Overall Confidence: {avg_conf:.3f}', fontsize=12, fontweight='bold')
    legend_y -= 0.06
    axes[1, 1].text(0.1, legend_y, f'Total Tiles: {len(predictions)}', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlim(0, 1); axes[1, 1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()
    plot_statistics(stats)
    if save_results:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        try:
            colored_bgr = cv2.cvtColor(colored_result, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_colored.jpg"), colored_bgr)
            prediction_bgr = cv2.cvtColor(prediction_map, cv2.COLOR_RGB2BGR)
            cv2.imwrite(os.path.join(output_dir, f"{base_name}_prediction_map.jpg"), prediction_bgr)
            print(f"💾 Results saved to {output_dir}/")
        except Exception as e:
            print(f"⚠️ Failed to save results: {e}")
    return {
        'original': image_rgb,
        'colored_result': colored_result,
        'prediction_map': prediction_map,
        'statistics': stats,
        'predictions': predictions,
        'confidences': confidences,
        'tiles': tiles,
        'positions': positions
    }

print("✅ Main processing function defined!")

# ------------------------------
# Utilities: Sample discovery & batch
# ------------------------------

def find_sample_images():
    """Find sample images in common directories"""
    available_images = []
    for search_dir in config.SAMPLE_PATHS:
        if os.path.exists(search_dir):
            print(f"📁 Checking {search_dir}...")
            try:
                image_extensions = ('.jpg', '.jpeg', '.png', '.tiff', '.tif')
                images = [os.path.join(search_dir, f) for f in os.listdir(search_dir)
                          if f.lower().endswith(image_extensions)]
                available_images.extend(images)
                print(f"  Found {len(images)} images")
            except Exception as e:
                print(f"  ⚠️ Error accessing directory: {e}")
    return available_images

def process_directory(input_dir, model, output_dir="batch_inference_results", max_images=5):
    """Process multiple images in a directory"""
    if model is None:
        print("❌ No model loaded")
        return
    if not os.path.exists(input_dir):
        print(f"❌ Directory not found: {input_dir}")
        return
    image_extensions = ('.jpg', '.jpeg', '.png', '.tiff', '.tif')
    image_files = [f for f in os.listdir(input_dir)
                   if f.lower().endswith(image_extensions)][:max_images]
    if not image_files:
        print(f"❌ No image files found in {input_dir}")
        return
    print(f"📁 Processing {len(image_files)} images from {input_dir}")
    os.makedirs(output_dir, exist_ok=True)
    all_stats = []
    for i, image_file in enumerate(image_files, 1):
        print(f"\n[{i}/{len(image_files)}] Processing {image_file}...")
        image_path = os.path.join(input_dir, image_file)
        try:
            results = process_image(
                image_path, model, save_results=True,
                output_dir=os.path.join(output_dir, f"image_{i:03d}")
            )
            if results:
                all_stats.append({'filename': image_file, 'stats': results['statistics']})
        except Exception as e:
            print(f"❌ Error processing {image_file}: {e}")
    if all_stats:
        print(f"\n📊 Batch Processing Summary for {len(all_stats)} images:")
        total_counts = {cls: 0 for cls in config.CLASS_LABELS}
        total_tiles = 0
        for stat in all_stats:
            for cls in config.CLASS_LABELS:
                total_counts[cls] += stat['stats'][cls]['count']
                total_tiles += stat['stats'][cls]['count']
        print(f"🔢 Total tiles processed: {total_tiles}")
        for cls, count in total_counts.items():
            percentage = count / total_tiles * 100 if total_tiles > 0 else 0
            print(f"  🔸 {cls.capitalize()}: {count} tiles ({percentage:.1f}%)")

# ------------------------------
# Confidence Analysis
# ------------------------------

def analyze_confidence(results):
    """Detailed confidence analysis"""
    if not results or 'confidences' not in results:
        print("❌ No results available for confidence analysis")
        return
    confidences = np.array(results['confidences'])
    predictions = np.array(results['predictions'])
    print("🔍 Model Confidence Analysis:")
    print(f"  📊 Average confidence: {np.mean(confidences):.3f}")
    print(f"  📊 Median confidence: {np.median(confidences):.3f}")
    print(f"  📊 Min confidence: {np.min(confidences):.3f}")
    print(f"  📊 Max confidence: {np.max(confidences):.3f}")
    print(f"  📊 Std deviation: {np.std(confidences):.3f}")
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes[0, 0].hist(confidences, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].axvline(np.mean(confidences), color='red', linestyle='--',
                       label=f'Mean: {np.mean(confidences):.3f}')
    axes[0, 0].set_xlabel('Confidence Score')
    axes[0, 0].set_ylabel('Number of Tiles')
    axes[0, 0].set_title('Overall Confidence Distribution')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    for i, class_name in enumerate(config.CLASS_LABELS):
        class_confidences = confidences[predictions == i]
        if len(class_confidences) > 0:
            color = tuple(c/255.0 for c in config.CLASS_COLORS[class_name])
            axes[0, 1].hist(class_confidences, bins=15, alpha=0.6,
                            label=f'{class_name} (n={len(class_confidences)})', color=color)
    axes[0, 1].set_xlabel('Confidence Score')
    axes[0, 1].set_ylabel('Number of Tiles')
    axes[0, 1].set_title('Confidence Distribution by Class')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    thresholds = np.arange(0.3, 1.0, 0.05)
    percentages = []
    for threshold in thresholds:
        high_conf_count = np.sum(confidences >= threshold)
        percentage = high_conf_count / len(confidences) * 100
        percentages.append(percentage)
    axes[1, 0].plot(thresholds, percentages, marker='o', linewidth=2, markersize=6)
    axes[1, 0].set_xlabel('Confidence Threshold')
    axes[1, 0].set_ylabel('% of Tiles Above Threshold')
    axes[1, 0].set_title('High Confidence Tiles Analysis')
    axes[1, 0].grid(True, alpha=0.3)
    class_conf_data = []
    class_labels = []
    for i, class_name in enumerate(config.CLASS_LABELS):
        class_confidences = confidences[predictions == i]
        if len(class_confidences) > 0:
            class_conf_data.append(class_confidences)
            class_labels.append(class_name)
    if class_conf_data:
        bp = axes[1, 1].boxplot(class_conf_data, labels=class_labels, patch_artist=True)
        for patch, class_name in zip(bp['boxes'], class_labels):
            color = tuple(c/255.0 for c in config.CLASS_COLORS[class_name])
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
    axes[1, 1].set_ylabel('Confidence Score')
    axes[1, 1].set_title('Confidence Distribution by Class (Box Plot)')
    axes[1, 1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    avg_confidence = float(np.mean(confidences))
    print("\n💡 Model Quality Assessment:")
    if avg_confidence > 0.9:
        print("  ✅ Excellent! Very high confidence in predictions.")
    elif avg_confidence > 0.8:
        print("  ✅ Good! High confidence in most predictions.")
    elif avg_confidence > 0.7:
        print("  ⚠️  Moderate confidence. Some predictions may be uncertain.")
    elif avg_confidence > 0.6:
        print("  ⚠️  Low confidence. Consider model improvements.")
    else:
        print("  ❌ Very low confidence. Model may need retraining.")
    low_conf_threshold = 0.7
    low_conf_count = int(np.sum(confidences < low_conf_threshold))
    if low_conf_count > 0:
        percentage = low_conf_count / len(confidences) * 100
        print(f"  📉 {low_conf_count} tiles ({percentage:.1f}%) have confidence < {low_conf_threshold}")
        for i, class_name in enumerate(config.CLASS_LABELS):
            class_low_conf = int(np.sum((predictions == i) & (confidences < low_conf_threshold)))
            if class_low_conf > 0:
                class_total = int(np.sum(predictions == i))
                class_percentage = class_low_conf / class_total * 100 if class_total > 0 else 0
                print(f"    🔸 {class_name}: {class_low_conf}/{class_total} tiles ({class_percentage:.1f}%)")

# ------------------------------
# CLI-like entry (optional)
# ------------------------------

def _maybe_demo_run():
    print("🔍 Searching for trained models...")
    model, model_type, model_path = find_and_load_model()
    if model is None:
        print("❌ No trained models found!")
        print("\n📝 Available model paths to check:")
        for name, path in config.MODEL_PATHS.items():
            exists = "✅" if os.path.exists(path) else "❌"
            print(f"  {exists} {name}: {path}")
        print("\n💡 Please train a model first or update the MODEL_PATHS in the config.")
        return
    print(f"✅ Successfully loaded {model_type} model from: {model_path}")
    print("🔍 Searching for sample images...")
    sample_images = find_sample_images()
    IMAGE_PATH = None
    if IMAGE_PATH is None and sample_images:
        IMAGE_PATH = sample_images[0]
        print(f"📸 Using sample image: {os.path.basename(IMAGE_PATH)}")
    if IMAGE_PATH and model is not None:
        print(f"\n🚀 Starting inference on: {IMAGE_PATH}")
        results = process_image(IMAGE_PATH, model, save_results=True)
        if results:
            print("\n✅ Inference completed successfully!")
            print(f"📊 Processed {len(results['predictions'])} tiles")
            print(f"🎯 Average confidence: {np.mean(results['confidences']):.3f}")
            analyze_confidence(results)
        else:
            print("❌ Inference failed")
    else:
        if IMAGE_PATH is None:
            print("❌ No image path specified. Please set IMAGE_PATH to your image file.")
            print("\n💡 Example:")
            print('IMAGE_PATH = "/path/to/your/image.jpg"')
        elif model is None:
            print("❌ No model loaded. Please train a model first.")

if __name__ == "__main__":
    # To run the demo pipeline (auto-detect model and sample), keep this True.
    RUN_DEMO = True
    if RUN_DEMO:
        _maybe_demo_run()
    else:
        print("Script imported. Use process_image(), process_directory(), analyze_confidence() as needed.")




✅ Libraries imported successfully!
✅ Model classes defined successfully!
Using device: cuda
GPU: Tesla V100-PCIE-16GB
✅ Configuration loaded successfully!
✅ Model loading functions defined!
✅ Image processing functions defined!
✅ Inference functions defined!
✅ Visualization functions defined!
✅ Analysis functions defined!
✅ Main processing function defined!
🔍 Searching for trained models...
❌ Error loading model: Error(s) in loading state_dict for RoofTextureClassifier:
	Missing key(s) in state_dict: "backbone.features.0.0.weight", "backbone.features.0.1.weight", "backbone.features.0.1.bias", "backbone.features.0.1.running_mean", "backbone.features.0.1.running_var", "backbone.features.1.0.block.0.0.weight", "backbone.features.1.0.block.0.1.weight", "backbone.features.1.0.block.0.1.bias", "backbone.features.1.0.block.0.1.running_mean", "backbone.features.1.0.block.0.1.running_var", "backbone.features.1.0.block.1.fc1.weight", "backbone.features.1.0.block.1.fc1.bias", "backbone.features.1

In [4]:
!ls

best_roof_classifier.pth    model_info.pkl	 training_history.png
checkpoints		    model_traning_v2.py
jupyter_training_runner.py  models
